# Test C: the Stage 1 x Stage 2 schema 2x2 under candidate injection, on the RTX 5090

**Upload both files into `work`:** this notebook and `injection_test_data.json`. Run cell 0 only if vLLM is not installed
(a fresh container), then **Kernel > Restart Kernel**, then run cells 1 onward in order.

In the paper's harness, switching the schema on changes **two** stages: Stage 1 expansion uses the schema prompt
(`EXPAND_QUERY_SCHEMA_V4`, its `expanded_query` field feeds Stage 2) and Stage 2 selection uses the flat schema.
The static-window 2x2 found most of Qwen's gain at Stage 1. This run repeats the 2x2 **under candidate injection**:
selection sees the 100 catalogue keywords SKYRAG's lexical ranker scores highest for the expanded query, instead of the
1,000-term window. Stage 1 is **not re-run**: the expansions from the static-window run are reused, and the candidates were
ranked from them on the laptop with SKYRAG's own ranker, so the two runs share Stage 1 exactly.

| | Stage 2 no schema | Stage 2 flat schema |
|---|---|---|
| **Stage 1 plain** | `arm2` (paper Arm 2) | `pv` |
| **Stage 1 schema** | `arm2_s1` | `pv_s1` (paper Arm 3) |

Serving matches the paper: LLaMA 3.1-8B at fp8, Qwen 2.5-7B at bf16. 50 queries x 2 platforms per condition.
Full replies are stored (no length cap). Final scoring is done on the laptop with SKYRAG's own scorer; the table here is a quick look.
**Pilot on this server only.** Download every `inject2x2_*` file before the session ends.

In [ ]:
# 0. Only on a fresh container: install vLLM, then restart the kernel
!pip install -U vllm
!pip uninstall -y torchaudio
!which vllm

In [1]:
# 1. Settings
N_QUERIES = 50
PASSES = 1                  # raise to 2 or 3 if time allows; each pass repeats every condition
RUNS = [("llama", "meta-llama/Llama-3.1-8B-Instruct", "fp8"),
        ("qwen",  "Qwen/Qwen2.5-7B-Instruct",         "bf16")]
CONDITIONS = [("arm2", "plain"), ("pv", "plain"), ("arm2_s1", "schema"), ("pv_s1", "schema")]
CONCURRENCY = 8
GPU_UTIL = "0.85"
MAX_LEN = "8192"

import os, sys, subprocess, time, json, shutil, signal, re, csv, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
csv.field_size_limit(10**8)

HERE = os.getcwd()
WORK = os.path.expanduser("~/work"); os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
PORT = 8000
VLLM = shutil.which("vllm") or "/opt/conda/bin/vllm"
PY = os.path.join(os.path.dirname(VLLM), "python")
_found = [c for c in (os.path.join(d, "injection_test_data.json") for d in (HERE, WORK, os.path.expanduser("~"))) if os.path.exists(c)]
if not _found:
    raise SystemExit("injection_test_data.json not found. Upload it next to this notebook.")
DATA = json.load(open(_found[0], encoding="utf-8"))
QUERIES = DATA["queries"][:N_QUERIES]
PLATFORMS = ["coursera", "edx"]
ELIGIBLE = {p: set(DATA["eligible"][p]) for p in PLATFORMS}
MAX_KW = DATA["max_keywords"]
BUDGET = 128 + 32 * MAX_KW            # the harness budget for Arm 2 and the flat schema alike
print(len(QUERIES), "queries | selection budget", BUDGET, "| candidates per call", len(DATA["candidates"]["qwen"]["plain"][QUERIES[0]["id"]]["coursera"]))

50 queries | selection budget 1408 | candidates per call 100


In [2]:
# 2. Hugging Face token (LLaMA is gated). Paste a READ token; kept in memory only.
from getpass import getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF read token: ").strip()
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

HF read token:  ········


NVIDIA GeForce RTX 5090, 3088 MiB, 32607 MiB


In [3]:
# 3. Helpers: server, requests, parsing
def healthy():
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def stop_server(p):
    if p is None:
        return
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM); p.wait(timeout=60)
    except Exception:
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception: pass
    time.sleep(5)

def start_server(model, precision, tag):
    args = [VLLM, "serve", model, "--port", str(PORT), "--dtype", "bfloat16", "--max-model-len", MAX_LEN,
            "--gpu-memory-utilization", GPU_UTIL, "--generation-config", "vllm"]
    if precision == "fp8":
        args += ["--quantization", "fp8"]
    log = open(os.path.join(WORK, f"inject2x2_server_{tag}.log"), "w")
    p = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while not healthy():
        if p.poll() is not None:
            raise SystemExit(f"Server failed; paste the last lines of inject2x2_server_{tag}.log to Claude.")
        if time.time() - t0 > 60 * 60:
            stop_server(p); raise SystemExit("Server start timed out.")
        time.sleep(5)
    print(f"  server ready after {(time.time() - t0) / 60:.1f} min")
    return p

def chat(model, prompt, max_tokens):
    body = {"model": model, "messages": [{"role": "user", "content": prompt}],
            "temperature": 0, "seed": 34, "max_tokens": max_tokens}
    req = urllib.request.Request(f"http://localhost:{PORT}/v1/chat/completions",
                                 json.dumps(body).encode(), {"Content-Type": "application/json"})
    t = time.time()
    try:
        r = json.load(urllib.request.urlopen(req, timeout=900))
    except urllib.error.HTTPError as e:
        return {"error": f"HTTP {e.code}: {e.read().decode(errors='replace')[:400]}", "seconds": time.time() - t}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}", "seconds": time.time() - t}
    ch = r["choices"][0]
    return {"text": ch["message"]["content"] or "", "finish": ch.get("finish_reason"),
            "completion_tokens": r["usage"]["completion_tokens"], "prompt_tokens": r["usage"]["prompt_tokens"],
            "seconds": time.time() - t}

def first_json(text):
    t = re.sub(r"^\s*```(?:json)?|```\s*$", "", text.strip(), flags=re.M).strip()
    for cand in (t, t[t.find("{"):t.rfind("}") + 1] if "{" in t else ""):
        try:
            return json.loads(cand)
        except Exception:
            pass
    return None

def s1_schema_ok(obj):
    # Mirrors QUERY_EXPANSION_SCHEMA_V4: exactly {expansions, expanded_query};
    # 2 to 4 string expansions; expanded_query a string.
    if not isinstance(obj, dict) or set(obj) != {"expansions", "expanded_query"}: return False
    ex = obj["expansions"]
    if not isinstance(ex, list) or not 2 <= len(ex) <= 4 or not all(isinstance(e, str) for e in ex): return False
    return isinstance(obj["expanded_query"], str)

def expand(model, q, mode):
    # The harness: under the schema, a valid reply's expanded_query is used; a violation falls back
    # to the raw reply text, exactly as Arm 2 would use it.
    if mode == "plain":
        r = chat(model, DATA["expand_prompt"].format(query=q["query"]), EXP_TOKENS)
        return (r.get("text") or "").strip() or q["query"], None
    prompt = DATA["expand_prompt_schema_v4"].format(query=q["query"], schema=DATA["expansion_schema_v4_rendered"])
    r = chat(model, prompt, EXP_TOKENS)
    text = (r.get("text") or "").strip()
    obj = first_json(text)
    if s1_schema_ok(obj):
        return obj["expanded_query"].strip() or q["query"], True
    return text or q["query"], False

def build(cond, platform, query, candidates):
    vocab = ", ".join(candidates)
    if cond.startswith("arm2"):
        return DATA["prompt_arm2"].format(platform=platform, vocabulary=vocab, query=query, max_keywords=MAX_KW)
    return DATA["prompt_arm3_v5"].format(platform=platform, vocabulary=vocab, query=query,
                                         max_keywords=MAX_KW, schema=DATA["schema_v5_rendered"])

print("ok")

ok


In [4]:
# 4. Run (saved after every condition; re-running skips what is already saved)
rows_path = os.path.join(WORK, "inject2x2_results.csv")
ROWS = list(csv.DictReader(open(rows_path, encoding="utf-8"))) if os.path.exists(rows_path) else []
done = {(r["family"], r["condition"], r["pass"]) for r in ROWS}

def save():
    with open(rows_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(ROWS[0])); w.writeheader(); w.writerows(ROWS)

for family, model, precision in RUNS:
    todo = [(c, s, k) for k in range(1, PASSES + 1) for c, s in CONDITIONS if (family, c, str(k)) not in done]
    if not todo:
        print(f"{family}: already done"); continue
    tag = f"{family}_{precision}"
    print(f"{tag}: starting server")
    server = start_server(model, precision, tag)
    try:
        for cond, s1, k in todo:
            def run_one(args):
                q, platform = args
                cands = DATA["candidates"][family][s1][q["id"]][platform]
                r = chat(model, build(cond, platform, DATA["expanded"][family][s1][q["id"]], cands), BUDGET)
                return {"family": family, "precision": precision, "condition": cond, "s1": s1, "pass": str(k),
                        "query_id": q["id"], "platform": platform, "finish": r.get("finish"),
                        "prompt_tokens": r.get("prompt_tokens"), "completion_tokens": r.get("completion_tokens"),
                        "seconds": round(r.get("seconds", 0), 1), "error": r.get("error", ""),
                        "raw": r.get("text") or ""}
            t = time.time()
            with ThreadPoolExecutor(CONCURRENCY) as ex:
                out = list(ex.map(run_one, [(q, p) for q in QUERIES for p in PLATFORMS]))
            ROWS += out; save()
            print(f"  {cond} pass {k}: {len(out)} calls in {(time.time() - t) / 60:.1f} min, "
                  f"{sum(r['finish'] == 'length' for r in out)} overruns, {sum(bool(r['error']) for r in out)} errors")
    finally:
        stop_server(server)
    print(f"{tag}: finished, saved")

llama_fp8: starting server
  server ready after 0.6 min
  arm2 pass 1: 100 calls in 0.7 min, 4 overruns, 0 errors
  pv pass 1: 100 calls in 0.7 min, 3 overruns, 0 errors
  arm2_s1 pass 1: 100 calls in 0.8 min, 15 overruns, 0 errors
  pv_s1 pass 1: 100 calls in 0.5 min, 3 overruns, 0 errors
llama_fp8: finished, saved
qwen_bf16: starting server
  server ready after 0.7 min
  arm2 pass 1: 100 calls in 0.7 min, 0 overruns, 0 errors
  pv pass 1: 100 calls in 1.0 min, 1 overruns, 0 errors
  arm2_s1 pass 1: 100 calls in 0.5 min, 2 overruns, 0 errors
  pv_s1 pass 1: 100 calls in 1.1 min, 7 overruns, 0 errors
qwen_bf16: finished, saved


In [5]:
# 5. Quick look (approximate: not-in-eligible counts as invalid). Final numbers come from the laptop rescore.
def quick(family, cond):
    byq = defaultdict(lambda: [0, 0])
    for r in ROWS:
        if (r["family"], r["condition"]) != (family, cond) or r["finish"] == "length" or r["error"]:
            continue
        obj = first_json(r["raw"])
        kw = obj.get("keywords") if isinstance(obj, dict) and isinstance(obj.get("keywords"), dict) else obj
        if not isinstance(kw, dict):
            continue
        if cond.startswith("pv") and not 1 <= len(kw) <= MAX_KW:
            continue
        byq[r["query_id"]][0] += len(kw)
        byq[r["query_id"]][1] += sum(k.strip().lower() not in ELIGIBLE[r["platform"]] for k in kw)
    vals = [b / n if n else 1.0 for n, b in (byq.get(q["id"], [0, 0]) for q in QUERIES)]
    return round(sum(vals) / len(vals), 4)

print("| model | arm2 | pv | arm2_s1 | pv_s1 |\n|---|---|---|---|---|")
for family, _, _ in RUNS:
    print(f"| {family} | " + " | ".join(str(quick(family, c)) for c, _ in CONDITIONS) + " |")
print("\ndownload:", [f for f in sorted(os.listdir(WORK)) if f.startswith("inject2x2_")])

| model | arm2 | pv | arm2_s1 | pv_s1 |
|---|---|---|---|---|
| llama | 0.1902 | 0.1947 | 0.0966 | 0.0792 |
| qwen | 0.2328 | 0.2534 | 0.0468 | 0.1779 |

download: ['inject2x2_results.csv', 'inject2x2_server_llama_fp8.log', 'inject2x2_server_qwen_bf16.log']


## Emergency stop

In [ ]:
subprocess.run("pkill -f 'vllm serve'", shell=True); time.sleep(5)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)